# Experiment1: classical vs time calibration on movielens


In this experiment we run svdpp on movielens dataset and compare different calibration methods

In [1]:
!pip install surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 10.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl size=2505168 sha256=b51333684771030d84b75665b4abb00027cf681a4a733aca4784fb21b6c2a58e
  Stored in directory: /root/.cache/pip/wheels/2a/8f/6e/7e2899163e2d85d8266daab4aa1cdabec7a6c56f83c015b5af
Successfully built scikit-surprise


In [2]:
import pandas as pd
from surprise.prediction_algorithms.matrix_factorization import SVD

In [8]:

from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import GridSearchCV



In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
from constants import *
from utils import load

## Load data

In [7]:
x_train = load(f"/content/drive/My Drive/{TRAIN_FEATURES_PATH}")
x_test  = load(f"/content/drive/My Drive/{TEST_FEATURES_PATH}")
x_val  = load(f"/content/drive/My Drive/{VAL_FEATURES_PATH}")


y_train = load(f"/content/drive/My Drive/{TRAIN_TARGET_PATH}")
y_test  = load(f"/content/drive/My Drive/{TEST_TARGET_PATH}")
y_val  = load(f"/content/drive/My Drive/{VAL_TARGET_PATH}")

In [10]:
x_train.rating.min(), x_train.rating.max()

(0.5, 5.0)

In [12]:
x_train.drop

,userId,movieId,rating
0,84327,50,5.0
1,84962,1844,4.5
2,81898,586,5.0
3,37585,36,3.5
4,48565,2890,4.0
...,...,...,...
14000179,66622,3296,4.0
14000180,100746,1704,3.5
14000181,15191,2717,3.5
14000182,92011,1276,5.0


In [14]:
reader = Reader(rating_scale=(0.5, 5))

train_data = pd.concat([x_train, y_train], axis=1)
val_data = pd.concat([x_val, y_val], axis=1)


In [22]:
x_train

,userId,movieId,rating
0,84327,50,5.0
1,84962,1844,4.5
2,81898,586,5.0
3,37585,36,3.5
4,48565,2890,4.0
...,...,...,...
14000179,66622,3296,4.0
14000180,100746,1704,3.5
14000181,15191,2717,3.5
14000182,92011,1276,5.0


In [21]:
train_data[['userId', 'movieId', 'rating']]

,userId,movieId,rating,rating
0,84327,50,5.0,5.0
1,84962,1844,4.5,4.5
2,81898,586,5.0,5.0
3,37585,36,3.5,3.5
4,48565,2890,4.0,4.0
...,...,...,...,...
14000179,66622,3296,4.0,4.0
14000180,100746,1704,3.5,3.5
14000181,15191,2717,3.5,3.5
14000182,92011,1276,5.0,5.0


In [20]:
Dataset.load_from_df(.reset_index(drop=True), reader=reader).build_full_trainset()

ValueError: too many values to unpack (expected 3)

In [13]:

trainset = Dataset.load_from_df(train_data[['userId', 'movieId', 'rating']], reader).build_full_trainset()
valset = Dataset.load_from_df(val_data[['userId', 'movieId', 'rating']], reader).build_full_trainset().build_testset()

ValueError: too many values to unpack (expected 3)

In [ ]:


# Define the parameter grid for hyperparameter tuning
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.1]
}

# Initialize GridSearchCV with SVD algorithm and precision as evaluation metric
gs = GridSearchCV(SVD, param_grid, measures=['precision'], cv=3)  # Adjust cv if needed

# Fit the GridSearchCV on the training data
gs.fit(Dataset.load_from_df(train_data[['user', 'item', 'rating']], reader))

# Print the best hyperparameters and corresponding precision
print(gs.best_score['precision'])
print(gs.best_params['precision'])

# Train the best model on the whole training set
best_svd = gs.best_estimator['precision']
best_svd.fit(trainset)

# Evaluate the best model on the validation set
predictions_val = best_svd.test(valset)
precision_val = accuracy.precision(predictions_val, k=5) # Adjust k as needed

print(f"Precision on validation set: {precision_val}")
